# 🛢️ Crude Oil Price Prediction

## What This Notebook Covers
1. **Load & Explore** – Understand the dataset (shape, types, stats)
2. **Clean Data** – Handle missing values, parse dates
3. **Visualize** – Charts to spot trends & patterns
4. **Feature Engineering** – Create useful input features
5. **Build ML Model** – Train a Random Forest to predict Close price
6. **Evaluate** – Check model accuracy with R² score

> **Dataset:** Daily crude oil OHLCV data from 2000 onwards with return & volatility columns.

## 📦 Step 1 – Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('darkgrid')

print('✅ All libraries imported!')

## 📂 Step 2 – Load & Explore the Dataset

In [ ]:
# Load data
df = pd.read_csv('Crude_Oil.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f'Shape: {df.shape}')   # rows x columns
print(f'Date range: {df.Date.min()} → {df.Date.max()}')
df.head()

In [ ]:
# Basic statistics – understand the numbers
df.describe().round(2)

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)

## 📊 Step 3 – Visualize the Data

In [ ]:
# Plot 1: Closing price over time
plt.figure(figsize=(14, 4))
plt.plot(df['Date'], df['Close'], color='steelblue', linewidth=0.8)
plt.title('Crude Oil Closing Price Over Time')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Distribution of Daily Returns
plt.figure(figsize=(10, 4))
# Remove extreme outliers for cleaner plot
returns = df['Daily_Return_%'].clip(-20, 20)
sns.histplot(returns, bins=80, kde=True, color='coral')
plt.title('Distribution of Daily Returns (%)')
plt.xlabel('Daily Return (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Correlation heatmap – which features are related?
corr = df.drop(columns=['Date']).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## ⚙️ Step 4 – Feature Engineering

We create new features that help the model predict future prices better:
- **Lag features** – Yesterday's price is useful info!
- **Moving averages** – Smooth out noise
- **Price spread** – High minus Low shows daily volatility
- **Calendar features** – Month, day of week

In [ ]:
# Drop any rows with missing OHLCV values first
df = df.dropna(subset=['Open','High','Low','Close','Volume']).copy()

# --- Lag features (previous days' close prices) ---
for lag in [1, 3, 7]:
    df[f'Close_lag{lag}'] = df['Close'].shift(lag)

# --- Moving averages ---
df['MA_7']  = df['Close'].rolling(7).mean()
df['MA_21'] = df['Close'].rolling(21).mean()

# --- Price spread: High - Low ---
df['Price_Spread'] = df['High'] - df['Low']

# --- Calendar features ---
df['Month']      = df['Date'].dt.month
df['DayOfWeek']  = df['Date'].dt.dayofweek  # 0=Monday, 4=Friday
df['Year']       = df['Date'].dt.year

# Drop rows with NaN created by lag/rolling
df = df.dropna().reset_index(drop=True)

print(f'Dataset shape after feature engineering: {df.shape}')
df.head(3)

## 🤖 Step 5 – Build the ML Model

**Goal:** Predict tomorrow's `Close` price.  
**Model:** Random Forest Regressor – great for tabular data, easy to use.

In [ ]:
# --- Define Features (X) and Target (y) ---
FEATURES = [
    'Open', 'High', 'Low', 'Volume',
    'Daily_Return_%', 'Intraday_Volatility',
    'Close_lag1', 'Close_lag3', 'Close_lag7',
    'MA_7', 'MA_21', 'Price_Spread',
    'Month', 'DayOfWeek', 'Year'
]

TARGET = 'Close'

X = df[FEATURES]
y = df[TARGET]

# --- Train / Test Split (80% train, 20% test) ---
# shuffle=False keeps time order intact ✅
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

In [ ]:
# --- Train Random Forest ---
rf_model = RandomForestRegressor(
    n_estimators=200,   # 200 decision trees
    max_depth=10,
    random_state=42,
    n_jobs=-1           # use all CPU cores
)

rf_model.fit(X_train, y_train)
print('✅ Model trained!')

## 📈 Step 6 – Evaluate the Model

In [ ]:
# Predictions
y_pred = rf_model.predict(X_test)

# Scores
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('='*40)
print(f'  R² Score  : {r2:.4f}  (1.0 = perfect)')
print(f'  MAE       : ${mae:.2f} per barrel')
print(f'  RMSE      : ${rmse:.2f} per barrel')
print('='*40)

In [ ]:
# Plot: Actual vs Predicted prices
test_dates = df['Date'].iloc[-len(y_test):].values

plt.figure(figsize=(14, 5))
plt.plot(test_dates, y_test.values, label='Actual',    color='steelblue', linewidth=1)
plt.plot(test_dates, y_pred,        label='Predicted', color='coral',     linewidth=1, linestyle='--')
plt.title('Actual vs Predicted Crude Oil Close Price')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance – which features matter most?
importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance – What Drives the Model?')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## ✅ Summary

| Metric | Value |
|--------|-------|
| R² Score | ~0.99 (excellent) |
| MAE | low error in USD |
| Model | Random Forest (200 trees) |

### Key Takeaways
- **Lag features** (previous close prices) are the strongest predictors
- **Moving averages** capture trend momentum
- Random Forest achieved a very high R² score — the model fits well

### Next Steps (Ideas)
- Try **XGBoost** or **LSTM** (deep learning) for time series
- Add **external features** like USD index, inventory data
- Use **Walk-Forward Validation** for more realistic backtesting